# Chapter 10: Creating Text Embedding Models - Hard Tasks

This notebook covers advanced multi-stage training pipelines: Augmented SBERT, TSDAE implementation, domain adaptation, and combined training strategies.

**Note on Code Organization**: Like in previous hard task notebooks, we use functions extensively because:
- **Reusability**: Call the same logic multiple times without copying code
- **Modularity**: Each function handles one specific task (train cross-encoder, label data, etc.)
- **Real-world practice**: Production systems always use functions for maintainability
- **Testing**: Easy to test individual components independently

## Setup

Run all cells in this section to set up the environment and load the model.

Before running these cells, review the concepts from the main Chapter 10 notebook (00_Start_Here.ipynb).

### [Optional] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab, uncomment and run the following code to install dependencies.

**Note**: Use a GPU for this notebook. In Google Colab, go to Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.

In [ ]:
# %%capture
# !pip install -q accelerate>=0.27.2 transformers>=4.38.2
# !pip install -q sentence-transformers>=3.0.0 datasets>=2.18.0
# !pip install -q nltk

### Model Loading

In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer, losses, InputExample, models
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.datasets import NoDuplicatesDataLoader

### Helper Functions

In [ ]:
def create_evaluator():
    """Create standard evaluator for consistent evaluation"""
    val_sts = load_dataset('glue', 'stsb', split='validation')
    return EmbeddingSimilarityEvaluator(
        sentences1=val_sts["sentence1"],
        sentences2=val_sts["sentence2"],
        scores=[score/5 for score in val_sts["label"]],
        main_similarity="cosine"
    )

## Challenges

Complete the following tasks by implementing the starter code.

### Level: Hard

**About This Task:**

Augmented SBERT uses a cross-encoder to create silver labels for additional training data. We break this into 5 functions, each handling one step of the pipeline.

#### Hard Task 1: Augmented SBERT Pipeline

### Instructions

1. Study the 5-stage pipeline (prepare gold data → train cross-encoder → create silver pairs → label with cross-encoder → train bi-encoder)
2. Run baseline bi-encoder on gold data only
3. Complete the silver data labeling function
4. Train bi-encoder on gold + silver
5. Compare results to see if augmentation helped

**Stage 1: Prepare Gold Data**

Gold data is our high-quality labeled dataset.

In [ ]:
def prepare_gold_data(num_samples=5_000):
    """
    Prepare gold dataset with ground-truth labels.
    
    Using a function here:
    - Can easily adjust data size
    - Consistent data preparation across experiments
    - Easy to swap datasets
    
    Args:
        num_samples: Number of examples to use
    
    Returns:
        gold_examples: List of InputExample for cross-encoder
        gold_df: Pandas DataFrame for easier data handling
    """
    print("Stage 1: Preparing gold data...")
    
    # Load MNLI data
    dataset = load_dataset("glue", "mnli", split="train").select(range(num_samples))
    
    # Convert to binary: entailment=1, neutral/contradiction=0
    mapping = {0: 1, 1: 0, 2: 0}
    
    # Create InputExample format for cross-encoder
    gold_examples = [
        InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
        for row in tqdm(dataset, desc="Creating gold examples")
    ]
    
    # Also create DataFrame for easier manipulation
    gold_df = pd.DataFrame({
        'sentence1': dataset['premise'],
        'sentence2': dataset['hypothesis'],
        'label': [mapping[label] for label in dataset['label']]
    })
    
    print(f"Gold dataset: {len(gold_examples)} examples")
    return gold_examples, gold_df

In [ ]:
# Create gold data
gold_examples, gold_df = prepare_gold_data(5_000)

**Stage 2: Train Cross-Encoder**

The cross-encoder will label our silver data.

In [ ]:
def train_cross_encoder(gold_examples):
    """
    Train a cross-encoder on gold data.
    
    Separating this into a function:
    - Can reuse trained cross-encoder
    - Easy to swap architectures
    - Clear separation of concerns
    
    Args:
        gold_examples: List of InputExample objects
    
    Returns:
        Trained CrossEncoder model
    """
    print("\nStage 2: Training cross-encoder...")
    
    # Create data loader
    gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)
    
    # Initialize cross-encoder
    cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2)
    
    # Train
    cross_encoder.fit(
        train_dataloader=gold_dataloader,
        epochs=1,
        show_progress_bar=True,
        warmup_steps=50,
        use_amp=False
    )
    
    print("Cross-encoder training complete")
    return cross_encoder

In [ ]:
# Train cross-encoder
cross_encoder = train_cross_encoder(gold_examples)

**Stage 3: Create Silver Data Pairs**

Generate new sentence pairs that need labels.

In [ ]:
def create_silver_pairs(start_idx=5_000, end_idx=15_000):
    """
    Create unlabeled silver dataset from a different data range.
    
    Using a function allows:
    - Easy control of data range
    - Consistent format with gold data
    - Reusability for different splits
    
    Args:
        start_idx: Starting index in dataset
        end_idx: Ending index
    
    Returns:
        pairs: List of (sentence1, sentence2) tuples
        silver_dataset: Dataset object for later use
    """
    print("\nStage 3: Creating silver data pairs...")
    
    silver = load_dataset("glue", "mnli", split="train").select(range(start_idx, end_idx))
    
    # Create pairs (no labels yet)
    pairs = list(zip(silver['premise'], silver['hypothesis']))
    
    print(f"Created {len(pairs)} silver pairs")
    return pairs, silver

In [ ]:
# Create silver pairs
silver_pairs, silver_raw = create_silver_pairs(5_000, 15_000)

**Stage 4: Label Silver Data with Cross-Encoder**

Your task: Complete this function to use the cross-encoder for labeling.

In [ ]:
def label_silver_data(cross_encoder, pairs, silver_dataset):
    """
    Use trained cross-encoder to label silver data.
    
    This function demonstrates the power of modular design:
    - Takes trained cross-encoder from Stage 2
    - Takes pairs from Stage 3
    - Produces labeled data for Stage 5
    - Each stage is independent and testable
    
    Args:
        cross_encoder: Trained CrossEncoder
        pairs: List of sentence pairs
        silver_dataset: Original dataset for reference
    
    Returns:
        silver_df: DataFrame with predicted labels
    """
    print("\nStage 4: Labeling silver data with cross-encoder...")
    
    # Fill in: Use cross_encoder.predict() with apply_softmax=True
    output = cross_encoder.predict(pairs, apply_softmax=True, show_progress_bar=True)
    
    # Take argmax to get predicted class
    predicted_labels = np.argmax(output, axis=1)
    
    # Create DataFrame
    silver_df = pd.DataFrame({
        "sentence1": silver_dataset["premise"],
        "sentence2": silver_dataset["hypothesis"],
        "label": predicted_labels
    })
    
    print(f"Labeled {len(silver_df)} silver examples")
    print(f"Label distribution: {silver_df['label'].value_counts().to_dict()}")
    
    return silver_df

In [ ]:
# Label silver data
silver_df = label_silver_data(cross_encoder, silver_pairs, silver_raw)

**Stage 5: Train Bi-Encoder on Combined Data**

Now we train a bi-encoder (SBERT) on both gold and silver data.

In [ ]:
def train_biencoder(combined_data, output_dir):
    """
    Train bi-encoder on gold + silver data.
    
    Separating this allows:
    - Training with different data combinations
    - A/B testing gold-only vs gold+silver
    - Reusing training logic
    
    Args:
        combined_data: DataFrame with all training data
        output_dir: Where to save model
    
    Returns:
        model: Trained SentenceTransformer
        results: Evaluation results
    """
    print(f"\nStage 5: Training bi-encoder on {len(combined_data)} examples...")
    
    # Remove duplicates
    combined_data = combined_data.drop_duplicates(subset=['sentence1', 'sentence2'], keep="first")
    
    # Convert to Dataset
    train_dataset = Dataset.from_pandas(combined_data, preserve_index=False)
    
    # Create model
    embedding_model = SentenceTransformer('bert-base-uncased')
    
    # Loss function
    train_loss = losses.CosineSimilarityLoss(model=embedding_model)
    
    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=32,
        warmup_steps=100,
        fp16=True,
        logging_steps=100,
    )
    
    # Evaluator
    evaluator = create_evaluator()
    
    # Train
    trainer = SentenceTransformerTrainer(
        model=embedding_model,
        args=args,
        train_dataset=train_dataset,
        loss=train_loss,
        evaluator=evaluator
    )
    
    trainer.train()
    
    # Evaluate
    results = evaluator(embedding_model)
    
    return embedding_model, results

Train with gold + silver data.

In [ ]:
# Combine gold and silver
combined = pd.concat([gold_df, silver_df], ignore_index=True, axis=0)

print(f"Combined dataset: {len(combined)} examples")
print(f"  Gold: {len(gold_df)}")
print(f"  Silver: {len(silver_df)}")

In [ ]:
# Train augmented model
augmented_model, augmented_results = train_biencoder(combined, "augmented_sbert")

print("\nAugmented SBERT Results:")
print(f"Spearman Cosine: {augmented_results['spearman_cosine']:.4f}")

### Task 1a: Train Baseline (Gold Only)

Compare against a model trained only on gold data.

In [ ]:
# Your task: Train bi-encoder on gold data only
baseline_model, baseline_results = train_biencoder(gold_df, "gold_only_sbert")

print("\nGold-Only Results:")
print(f"Spearman Cosine: {baseline_results['spearman_cosine']:.4f}")

Compare results.

In [ ]:
print("\n" + "="*60)
print("Augmented SBERT Comparison:")
print(f"Gold only:        {baseline_results['spearman_cosine']:.4f}")
print(f"Gold + Silver:    {augmented_results['spearman_cosine']:.4f}")
print(f"Improvement:      {augmented_results['spearman_cosine'] - baseline_results['spearman_cosine']:.4f}")

### Questions

1. Did silver data improve performance? Why would imperfect labels still help?

2. Which stage took the longest? How would you optimize this pipeline for production?

3. Why use 5 separate functions instead of one big script? Give 3 specific advantages.

**About This Task:**

TSDAE trains embeddings without labels by learning to reconstruct sentences from noisy versions. We build this using modular functions for data preparation, model creation, and training.

#### Hard Task 2: TSDAE Implementation

### Instructions

1. Study the 4-function pipeline (prepare sentences → add noise → create model → train with DAE loss)
2. Run TSDAE with default noise (deletion)
3. Implement custom noise function (word shuffling)
4. Compare TSDAE vs supervised training
5. Analyze when unsupervised learning helps

In [ ]:
# Download NLTK tokenizer
import nltk
nltk.download('punkt', quiet=True)

**Function 1: Prepare Sentences**

Extract unique sentences for unsupervised training.

In [ ]:
def prepare_sentences(num_samples=10_000):
    """
    Extract unique sentences for TSDAE training.
    
    Using a function:
    - Easy to control data size
    - Can swap data sources
    - Ensures deduplication
    
    Args:
        num_samples: Number of examples to load
    
    Returns:
        List of unique sentences
    """
    print("Preparing sentences for TSDAE...")
    
    # Load MNLI
    mnli = load_dataset("glue", "mnli", split="train").select(range(num_samples))
    
    # Combine premise and hypothesis into one flat list
    flat_sentences = mnli["premise"] + mnli["hypothesis"]
    
    # Remove duplicates
    unique_sentences = list(set(flat_sentences))
    
    print(f"Collected {len(unique_sentences)} unique sentences")
    return unique_sentences

In [ ]:
sentences = prepare_sentences(10_000)

**Function 2: Add Noise to Sentences**

Create damaged versions for the denoising task.

In [ ]:
from sentence_transformers.datasets import DenoisingAutoEncoderDataset

def create_noisy_data(sentences, noise_fn=None):
    """
    Create noisy versions of sentences.
    
    Separating noise creation:
    - Can test different noise strategies
    - Easy to visualize noise effects
    - Allows custom noise functions
    
    Args:
        sentences: List of clean sentences
        noise_fn: Optional custom noise function
    
    Returns:
        damaged_data: DenoisingAutoEncoderDataset
        dataset: Dataset for training
    """
    print("\nAdding noise to sentences...")
    
    if noise_fn:
        damaged_data = DenoisingAutoEncoderDataset(sentences, noise_fn=noise_fn)
    else:
        # Default: delete words with 60% probability
        damaged_data = DenoisingAutoEncoderDataset(sentences)
    
    # Convert to Dataset format
    train_dataset = {"damaged_sentence": [], "original_sentence": []}
    for data in tqdm(damaged_data, desc="Processing noisy data"):
        train_dataset["damaged_sentence"].append(data.texts[0])
        train_dataset["original_sentence"].append(data.texts[1])
    
    dataset = Dataset.from_dict(train_dataset)
    
    print(f"Created {len(dataset)} noisy examples")
    return damaged_data, dataset

In [ ]:
# Create noisy data with default deletion
damaged_data, noisy_dataset = create_noisy_data(sentences)

View examples of noise.

In [ ]:
print("Noise examples:")
for i in range(3):
    print(f"\nExample {i+1}:")
    print(f"  Original: {noisy_dataset[i]['original_sentence']}")
    print(f"  Damaged:  {noisy_dataset[i]['damaged_sentence']}")

**Function 3: Create TSDAE Model**

Build model with encoder-decoder architecture.

In [ ]:
def create_tsdae_model():
    """
    Create embedding model with CLS pooling for TSDAE.
    
    Using a function:
    - Consistent model architecture
    - Easy to swap base models
    - Clear separation from training
    
    Returns:
        SentenceTransformer model with CLS pooling
    """
    print("\nCreating TSDAE model...")
    
    # Create model components
    word_embedding_model = models.Transformer('bert-base-uncased')
    pooling_model = models.Pooling(
        word_embedding_model.get_word_embedding_dimension(),
        'cls'  # TSDAE uses CLS token
    )
    
    # Combine into SentenceTransformer
    model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
    
    print("Model created with CLS pooling")
    return model

In [ ]:
tsdae_model = create_tsdae_model()

**Function 4: Train with TSDAE Loss**

Train model to reconstruct sentences from noisy input.

In [ ]:
def train_tsdae(model, dataset, output_dir):
    """
    Train model with denoising autoencoder loss.
    
    This function shows the TSDAE training pattern:
    - Takes noisy dataset from Function 2
    - Uses DenoisingAutoEncoderLoss
    - Evaluates on downstream task
    
    Args:
        model: SentenceTransformer with CLS pooling
        dataset: Dataset with damaged and original sentences
        output_dir: Where to save model
    
    Returns:
        model: Trained model
        results: Evaluation results
    """
    print("\nTraining with TSDAE loss...")
    
    # Create DAE loss
    train_loss = losses.DenoisingAutoEncoderLoss(
        model,
        tie_encoder_decoder=True
    )
    
    # Move decoder to GPU
    train_loss.decoder = train_loss.decoder.to("cuda")
    
    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=16,
        warmup_steps=100,
        fp16=True,
        logging_steps=100,
    )
    
    # Create evaluator
    evaluator = create_evaluator()
    
    # Train
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        loss=train_loss,
        evaluator=evaluator
    )
    
    trainer.train()
    
    # Evaluate
    results = evaluator(model)
    
    return model, results

In [ ]:
# Train TSDAE model
tsdae_model, tsdae_results = train_tsdae(tsdae_model, noisy_dataset, "tsdae_model")

print("\nTSDAE Results:")
print(f"Spearman Cosine: {tsdae_results['spearman_cosine']:.4f}")

### Task 2a: Custom Noise Function

Your task: Implement word shuffling noise.

In [ ]:
import random

def shuffle_words(text):
    """
    Custom noise: shuffle word order.
    
    Args:
        text: Original sentence
    
    Returns:
        Shuffled sentence
    """
    # Fill in: Split text into words, shuffle, rejoin
    words = text.split()
    random.shuffle(words)
    return ' '.join(words)

# Test the noise function
test_sentence = "The quick brown fox jumps over the lazy dog"
print(f"Original: {test_sentence}")
print(f"Shuffled: {shuffle_words(test_sentence)}")

In [ ]:
# Create dataset with custom noise
_, shuffled_dataset = create_noisy_data(sentences[:5000], noise_fn=shuffle_words)

print("\nShuffle noise examples:")
for i in range(2):
    print(f"\nExample {i+1}:")
    print(f"  Original: {shuffled_dataset[i]['original_sentence']}")
    print(f"  Shuffled: {shuffled_dataset[i]['damaged_sentence']}")

### Questions

1. How does TSDAE learn good embeddings without labels?

2. Compare deletion vs shuffling noise. Which is harder for the model to denoise?

3. When would you use TSDAE instead of supervised training?

**About This Task:**

Domain adaptation fine-tunes a general embedding model on domain-specific data to improve performance on that domain. We use a multi-stage pipeline.

#### Hard Task 3: Domain Adaptation Pipeline

### Instructions

1. Evaluate a general-purpose model on a domain-specific task
2. Collect domain-specific unlabeled data
3. Fine-tune with domain data using TSDAE
4. Re-evaluate on the domain task
5. Measure improvement from domain adaptation

**Stage 1: Baseline Evaluation**

Test a general model on a specialized domain.

In [ ]:
def evaluate_on_domain(model, task_name="Banking77Classification"):
    """
    Evaluate model on domain-specific task.
    
    Using a function:
    - Consistent evaluation across experiments
    - Easy to swap tasks
    - Track performance over time
    
    Args:
        model: SentenceTransformer to evaluate
        task_name: MTEB task name
    
    Returns:
        accuracy: Main score from evaluation
    """
    from mteb import MTEB
    
    print(f"\nEvaluating on {task_name}...")
    
    evaluation = MTEB(tasks=[task_name])
    results = evaluation.run(model)
    
    accuracy = results[0]['scores']['test'][0]['main_score']
    print(f"Accuracy: {accuracy:.4f}")
    
    return accuracy

In [ ]:
# Load general-purpose model
general_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Evaluate baseline
baseline_accuracy = evaluate_on_domain(general_model)

**Stage 2: Collect Domain Data**

Your task: Prepare domain-specific sentences (we'll use banking domain).

In [ ]:
def collect_domain_data(domain="banking", num_samples=5_000):
    """
    Collect domain-specific unlabeled sentences.
    
    In production:
    - Scrape domain documents
    - Use domain-specific corpora
    - Extract from task data (without labels)
    
    Args:
        domain: Domain name
        num_samples: Number of sentences
    
    Returns:
        List of domain sentences
    """
    print(f"\nCollecting {domain} domain data...")
    
    # For this example, use Banking77 training text (unlabeled)
    banking_data = load_dataset("banking77", split="train").select(range(num_samples))
    
    # Extract just the text (ignore labels)
    domain_sentences = list(set(banking_data["text"]))
    
    print(f"Collected {len(domain_sentences)} unique domain sentences")
    return domain_sentences

In [ ]:
# Collect banking domain data
domain_sentences = collect_domain_data("banking", 5_000)

**Stage 3: Domain Adaptation with TSDAE**

Fine-tune on domain data using unsupervised TSDAE.

In [ ]:
def domain_adapt(model, domain_sentences, output_dir):
    """
    Adapt model to domain using TSDAE.
    
    This combines previous functions:
    - Uses noisy data creation from Task 2
    - Uses TSDAE training from Task 2
    - Applied to domain-specific data
    
    Args:
        model: Pre-trained SentenceTransformer
        domain_sentences: Domain-specific sentences
        output_dir: Where to save adapted model
    
    Returns:
        Adapted model
    """
    print("\nAdapting to domain...")
    
    # Create noisy data
    _, noisy_domain = create_noisy_data(domain_sentences)
    
    # Create DAE loss
    train_loss = losses.DenoisingAutoEncoderLoss(
        model,
        tie_encoder_decoder=True
    )
    train_loss.decoder = train_loss.decoder.to("cuda")
    
    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=16,
        warmup_steps=50,
        fp16=True,
        logging_steps=50,
    )
    
    # Train
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=noisy_domain,
        loss=train_loss,
    )
    
    trainer.train()
    
    print("Domain adaptation complete")
    return model

In [ ]:
# Adapt model to banking domain
adapted_model = domain_adapt(general_model, domain_sentences, "domain_adapted")

**Stage 4: Re-evaluate on Domain Task**

In [ ]:
# Evaluate adapted model
adapted_accuracy = evaluate_on_domain(adapted_model)

Compare results.

In [ ]:
print("\n" + "="*60)
print("Domain Adaptation Results:")
print(f"Baseline (general):  {baseline_accuracy:.4f}")
print(f"Adapted (banking):   {adapted_accuracy:.4f}")
print(f"Improvement:         {adapted_accuracy - baseline_accuracy:.4f}")

### Questions

1. Did domain adaptation improve accuracy? Why would unsupervised adaptation help?

2. What makes a good domain for adaptation? When would adaptation not help?

3. Could you combine domain adaptation with supervised fine-tuning? How?

**About This Task:**

Real-world systems often combine multiple training strategies. This task chains TSDAE pretraining, supervised fine-tuning, and iterative evaluation.

#### Hard Task 4: Multi-Stage Training Pipeline

### Instructions

1. Stage 1: Pretrain with TSDAE on large unlabeled corpus
2. Stage 2: Fine-tune on labeled data
3. Stage 3: Evaluate and identify weak points
4. Stage 4: Augment data for weak points
5. Stage 5: Final fine-tuning and evaluation

**Stage 1: TSDAE Pretraining**

In [ ]:
def stage1_pretrain():
    """
    Stage 1: Unsupervised pretraining with TSDAE.
    
    Returns:
        Pretrained model
    """
    print("=" * 60)
    print("STAGE 1: TSDAE Pretraining")
    print("=" * 60)
    
    # Prepare sentences
    sentences = prepare_sentences(5_000)
    
    # Add noise
    _, noisy_data = create_noisy_data(sentences)
    
    # Create model
    model = create_tsdae_model()
    
    # Train
    model, _ = train_tsdae(model, noisy_data, "stage1_pretrain")
    
    return model

In [ ]:
# Run Stage 1
pretrained_model = stage1_pretrain()

**Stage 2: Supervised Fine-tuning**

In [ ]:
def stage2_supervised(model):
    """
    Stage 2: Fine-tune on labeled data.
    
    Args:
        model: Pretrained model from Stage 1
    
    Returns:
        Fine-tuned model, evaluation results
    """
    print("\n" + "=" * 60)
    print("STAGE 2: Supervised Fine-tuning")
    print("=" * 60)
    
    # Prepare labeled data
    train_data = load_dataset("glue", "mnli", split="train").select(range(5_000))
    train_data = train_data.filter(lambda x: x['label'] == 0)
    
    dataset = Dataset.from_dict({
        "anchor": train_data["premise"],
        "positive": train_data["hypothesis"],
    })
    
    # Loss function
    loss = losses.MultipleNegativesRankingLoss(model=model)
    
    # Training arguments
    args = SentenceTransformerTrainingArguments(
        output_dir="stage2_supervised",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        warmup_steps=50,
        fp16=True,
        logging_steps=50,
    )
    
    # Evaluator
    evaluator = create_evaluator()
    
    # Train
    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        loss=loss,
        evaluator=evaluator
    )
    
    trainer.train()
    
    # Evaluate
    results = evaluator(model)
    
    print(f"\nStage 2 Score: {results['spearman_cosine']:.4f}")
    return model, results

In [ ]:
# Run Stage 2
finetuned_model, stage2_results = stage2_supervised(pretrained_model)

### Task 4a: Compare Against Baseline

Your task: Train a model without TSDAE pretraining and compare.

In [ ]:
# Fill in: Train model with only supervised learning (skip Stage 1)
# This shows the value of TSDAE pretraining

def baseline_supervised():
    """Train model with only supervised learning, no TSDAE pretraining"""
    print("\n" + "=" * 60)
    print("BASELINE: Supervised Only (No TSDAE)")
    print("=" * 60)
    
    # Create fresh model
    model = SentenceTransformer('bert-base-uncased')
    
    # Train with supervised only
    model, results = stage2_supervised(model)
    
    return model, results

# Uncomment to run:
# baseline_model, baseline_results = baseline_supervised()

Compare multi-stage vs single-stage training.

In [ ]:
# If you ran baseline, compare:
# print("\n" + "="*60)
# print("Multi-Stage Training Comparison:")
# print(f"Supervised only:     {baseline_results['spearman_cosine']:.4f}")
# print(f"TSDAE + Supervised:  {stage2_results['spearman_cosine']:.4f}")
# print(f"Improvement:         {stage2_results['spearman_cosine'] - baseline_results['spearman_cosine']:.4f}")

### Questions

1. Did TSDAE pretraining improve the final supervised model?

2. Why break training into stages instead of one long training run?

3. What other stages could you add? (e.g., domain adaptation, hard negative mining, etc.)